# Crew Exercise: Week 3 (`3_crew`)

This exercise stays in Week 3 scope (CrewAI templates and patterns) and implements **something new**:
a custom multi-agent crew for **career pivot planning**.


## Week 3 Concept Consolidation

From `3_crew/` core projects (`coder`, `debate`, `engineering_team`, `financial_researcher`, `stock_picker`):

1. `config/agents.yaml` defines agent roles/goals/backstory/model.
2. `config/tasks.yaml` defines task descriptions, expected output, context, and output files.
3. `crew.py` wires agents + tasks with `@CrewBase`, `@agent`, `@task`, and `@crew`.
4. `main.py` supplies runtime inputs and runs `.kickoff(inputs=...)`.
5. Processes include `Process.sequential` and `Process.hierarchical` with manager delegation.
6. Tools/memory are optional enhancements used by some crews.


## Exercise Build: `career_pivot_crew`

This notebook scaffolds a new Week 3-style CrewAI project under:

`3_crew/community_contributions/eddy_mmaitsimwale/career_pivot_crew`

Crew design:
- `market_researcher`
- `skills_gap_analyst`
- `roadmap_writer`
- `manager`


In [ ]:
from pathlib import Path

base = Path('3_crew/community_contributions/eddy_mmaitsimwale/career_pivot_crew')
config_dir = base / 'src' / 'career_pivot_crew' / 'config'

for d in [config_dir, base / 'output', base / 'knowledge']:
    d.mkdir(parents=True, exist_ok=True)

(base / 'knowledge' / 'user_preference.txt').write_text(
    'Focus on practical, affordable upskilling paths and realistic timelines.',
    encoding='utf-8'
)

print('Created:', base)


In [ ]:
agents_yaml_lines = [
    "market_researcher:",
    "  role: >",
    "    Career Market Researcher",
    "  goal: >",
    "    Research current job-market demand, role expectations, and salary bands for {target_role} in {region}.",
    "  backstory: >",
    "    You are a practical labor-market analyst who summarizes role demand and hiring trends.",
    "  llm: openai/gpt-4o-mini",
    "",
    "skills_gap_analyst:",
    "  role: >",
    "    Skills Gap Analyst",
    "  goal: >",
    "    Compare the candidate background against requirements for {target_role} and identify skill gaps.",
    "  backstory: >",
    "    You are an experienced career coach and technical assessor.",
    "  llm: openai/gpt-4o-mini",
    "",
    "roadmap_writer:",
    "  role: >",
    "    Career Roadmap Planner",
    "  goal: >",
    "    Build a realistic 90-day learning and project roadmap toward {target_role}.",
    "  backstory: >",
    "    You specialize in creating actionable week-by-week learning plans.",
    "  llm: openai/gpt-4o-mini",
    "",
    "manager:",
    "  role: >",
    "    Program Manager",
    "  goal: >",
    "    Coordinate all agents to produce one coherent career transition package.",
    "  backstory: >",
    "    You are a detail-oriented manager who delegates tasks and consolidates outputs.",
    "  llm: openai/gpt-4o",
]

tasks_yaml_lines = [
    "research_market:",
    "  description: >",
    "    Research demand for {target_role} in {region}. Summarize required skills, responsibilities,",
    "    salary ranges, and market outlook.",
    "  expected_output: >",
    "    Concise role market report with bullet points and credible signals.",
    "  agent: market_researcher",
    "  output_file: output/market_report.md",
    "",
    "analyze_skills_gap:",
    "  description: >",
    "    Using the candidate profile and market report, identify strengths, missing skills, and priority gaps.",
    "  expected_output: >",
    "    Skills gap analysis with high/medium/low priority recommendations.",
    "  agent: skills_gap_analyst",
    "  context:",
    "    - research_market",
    "  output_file: output/skills_gap.md",
    "",
    "write_transition_plan:",
    "  description: >",
    "    Produce a 90-day transition plan with weekly milestones, project ideas, interview prep actions,",
    "    and application strategy.",
    "  expected_output: >",
    "    A practical 90-day action plan that the user can follow immediately.",
    "  agent: roadmap_writer",
    "  context:",
    "    - analyze_skills_gap",
    "  output_file: output/transition_plan.md",
]

agents_yaml = "\n".join(agents_yaml_lines) + "\n"
tasks_yaml = "\n".join(tasks_yaml_lines) + "\n"

(config_dir / 'agents.yaml').write_text(agents_yaml, encoding='utf-8')
(config_dir / 'tasks.yaml').write_text(tasks_yaml, encoding='utf-8')

print('Wrote config files')


In [ ]:
crew_py_lines = [
    'from crewai import Agent, Crew, Process, Task',
    'from crewai.project import CrewBase, agent, crew, task',
    '',
    '',
    '@CrewBase',
    'class CareerPivotCrew:',
    "    agents_config = 'config/agents.yaml'",
    "    tasks_config = 'config/tasks.yaml'",
    '',
    '    @agent',
    '    def market_researcher(self) -> Agent:',
    "        return Agent(config=self.agents_config['market_researcher'], verbose=True)",
    '',
    '    @agent',
    '    def skills_gap_analyst(self) -> Agent:',
    "        return Agent(config=self.agents_config['skills_gap_analyst'], verbose=True)",
    '',
    '    @agent',
    '    def roadmap_writer(self) -> Agent:',
    "        return Agent(config=self.agents_config['roadmap_writer'], verbose=True)",
    '',
    '    @task',
    '    def research_market(self) -> Task:',
    "        return Task(config=self.tasks_config['research_market'])",
    '',
    '    @task',
    '    def analyze_skills_gap(self) -> Task:',
    "        return Task(config=self.tasks_config['analyze_skills_gap'])",
    '',
    '    @task',
    '    def write_transition_plan(self) -> Task:',
    "        return Task(config=self.tasks_config['write_transition_plan'])",
    '',
    '    @crew',
    '    def crew(self) -> Crew:',
    "        manager = Agent(config=self.agents_config['manager'], allow_delegation=True)",
    '        return Crew(',
    '            agents=self.agents,',
    '            tasks=self.tasks,',
    '            process=Process.hierarchical,',
    '            manager_agent=manager,',
    '            verbose=True,',
    '        )',
]

main_py_lines = [
    '#!/usr/bin/env python',
    'from career_pivot_crew.crew import CareerPivotCrew',
    '',
    '',
    'def run():',
    '    inputs = {',
    "        'target_role': 'AI Engineer',",
    "        'region': 'East Africa',",
    '    }',
    '    result = CareerPivotCrew().crew().kickoff(inputs=inputs)',
    '    print(result.raw)',
    '',
    '',
    "if __name__ == '__main__':",
    '    run()',
]

pyproject_toml_lines = [
    '[project]',
    'name = "career_pivot_crew"',
    'version = "0.1.0"',
    'description = "Week 3 CrewAI exercise - career pivot planner"',
    'readme = "README.md"',
    'requires-python = ">=3.10,<3.13"',
    'dependencies = [',
    '  "crewai[tools]>=0.95.0"',
    ']',
    '',
    '[tool.crewai]',
    'type = "crew"',
]

readme_lines = [
    '# Career Pivot Crew',
    '',
    'A Week 3 CrewAI exercise project implementing a new custom crew.',
    '',
    '## Run',
    '',
    '1. Add `OPENAI_API_KEY` to `.env`',
    '2. Install dependencies:',
    '   - `uv sync` (or `crewai install`)',
    '3. Run:',
    '   - `crewai run`',
    '',
    '## Outputs',
    '',
    '- `output/market_report.md`',
    '- `output/skills_gap.md`',
    '- `output/transition_plan.md`',
]

crew_py = "\n".join(crew_py_lines) + "\n"
main_py = "\n".join(main_py_lines) + "\n"
pyproject_toml = "\n".join(pyproject_toml_lines) + "\n"
readme_md = "\n".join(readme_lines) + "\n"

src_dir = base / 'src' / 'career_pivot_crew'
src_dir.mkdir(parents=True, exist_ok=True)

(src_dir / '__init__.py').write_text('', encoding='utf-8')
(src_dir / 'crew.py').write_text(crew_py, encoding='utf-8')
(src_dir / 'main.py').write_text(main_py, encoding='utf-8')
(base / 'pyproject.toml').write_text(pyproject_toml, encoding='utf-8')
(base / 'README.md').write_text(readme_md, encoding='utf-8')

print('Wrote source files')


In [ ]:
files = sorted(str(p) for p in base.rglob('*') if p.is_file())
for f in files:
    print(f)


## What Is New

- New use case: career pivot planning.
- New domain-specific agents/tasks in Week 3 CrewAI format.
- Hierarchical manager process for coordination.
